In [3]:
import urllib.request as urllib2
prefix = 'https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/embed'

urllib2.urlretrieve(f"{prefix}/download.py", "download.py")
urllib2.urlretrieve(f"{prefix}/embedder.py", "embedder.py")
print("Done! Both files are now in your project directory.")

Done! Both files are now in your project directory.


In [4]:
import os

# 1. See where the notebook is currently running
print("Current locked folder:", os.getcwd())

# 2. Force the notebook to switch to your new directory
new_path = r"C:\Users\user\PycharmProjects\LLMops_2026\02_vector_search"

if os.path.exists(new_path):
    os.chdir(new_path)
    print("✅ Successfully switched to:", os.getcwd())
else:
    print("❌ Could not find that folder path. Please double check the spelling!")

Current locked folder: C:\Users\user\PycharmProjects\LLMops_2026\02_vector_search
✅ Successfully switched to: C:\Users\user\PycharmProjects\LLMops_2026\02_vector_search


**Q1. Embedding a query**

In [5]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [15]:
# Print the keys of the very first document dictionary
print(documents[0].keys())

dict_keys(['content', 'filename'])


In [6]:
from embedder import Embedder
model = Embedder()
text_to_embed = "How does approximate nearest neighbor search work?"
vectors = model.encode(text_to_embed)
# Print the very first value (v[0])
print("The first value (v[0]) is:", vectors[0])


The first value (v[0]) is: -0.02058203437252893


**Q2. Cosine similarity**

In [7]:
import numpy as np
target_doc = None
for doc in documents:
    if "07-sqlitesearch-vector.md" in doc["filename"]:
        target_doc = doc
        break
if target_doc is None:
    raise ValueError("Could not find a suitable document.")
v_doc = model.encode(target_doc["content"])
vq1 = vectors
dot_product = np.dot(vq1,v_doc)
norm_q1 = np.linalg.norm(vq1)
norm_doc = np.linalg.norm(v_doc)

cosine_similarity = dot_product / (norm_q1 * norm_doc)

print("The cosine similarity is:", cosine_similarity)


The cosine similarity is: 0.361070272255897


**Q3. Chunking and search by hand**

In [8]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

In [9]:
chunk_text = [chunk['content'] for chunk in chunks]
chunk_vector = model.encode_batch(chunk_text)
X = np.array(chunk_vector)

In [10]:
scores = X.dot(vq1)

In [11]:
highest = np.argmax(scores)
best_chunk = chunks[highest]
print("Highest score:", scores[highest])
print("Belongs to file:", best_chunk['filename'])

Highest score: 0.6489017718578813
Belongs to file: 02-vector-search/lessons/07-sqlitesearch-vector.md


**Q4. Vector search with minsearch**

In [12]:
from minsearch import VectorSearch
vindex = VectorSearch(keyword_fields = ['course'])
vindex.fit(X,chunks)

query = "What metric do we use to evaluate a search engine?"
query_vector = model.encode(query)

results = vindex.search(query_vector, num_results=5)
results[0]

{'start': 0,
 'content': "# Search Evaluation Metrics\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=TuirMy3Pdbk&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we computed relevance lists for search results.\nWe can turn those lists into metrics.\n\n## Hit Rate\n\nHit Rate (also called Recall@k) measures the fraction of queries where\nthe correct document appears anywhere in the results:\n\n```python\nexample = [\n    [1, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 1, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n]\n```\n\nEach line is one query. If a line contains `1`, search found the\ncorrect document somewhere in the top 5 results. If the line contains\nonly zeros, search did not find the correct document.\n\nIn our setup

**Q5. Text search vs vector search**

In [13]:
for chunk in chunks:
    chunk['course'] = 'llm-zoomcamp'

In [14]:
from minsearch import VectorSearch,Index

# --- 1. TEXT SEARCH SETUP ---
text_index = Index(text_fields = ['content'],keyword_fields = ['course'])
text_index.fit(chunks)
text_res = text_index.search(query ="How do I store vectors in PostgreSQL?",
                             filter_dict = {"course": "llm-zoomcamp"},
                             num_results = 5)

# --- 2. VECTOR SEARCH SETUP ---
vindex = VectorSearch(keyword_fields = ['course'])
vindex.fit(X,chunks)

query_vector = model.encode("How do I store vectors in PostgreSQL?")

vector_results = vindex.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

print("=== TOP TEXT SEARCH MATCH ===")
if text_res:
    print(f"File: {text_res[0]['filename']}\n")
    print(text_res[0]['content'][:300] + "...\n")
else:
    print("No text results found.\n")

print("=== TOP VECTOR SEARCH MATCH ===")
if vector_results:
    print(f"File: {vector_results[0]['filename']}\n")
    print(vector_results[0]['content'][:300] + "...")
else:
    print("No vector results found.")

=== TOP TEXT SEARCH MATCH ===
File: 02-vector-search/lessons/02-embeddings.md

get 0.01.

The first score for `q1` vs `d` (0.32) is higher, so that query is more
similar to the document about registration. The second score for `q2`
vs `d` sits near 0, because installing Docker has nothing to do with
registration. A score near 0 means the two vectors are about as
different as t...

=== TOP VECTOR SEARCH MATCH ===
File: 02-vector-search/lessons/08-pgvector.md

# Vector Search with PGVector

Video: [Watch this lesson](https://www.youtube.com/watch?v=0P54MFyz-mc&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

Many real databases can do vector search. Elasticsearch has it, and
there are dedicated stores like Qdrant and Chroma. We'll go with
Postgres. Most of us al...


In [15]:
top_text_files = [doc['filename'] for doc in text_res[:5]]
top_vector_files = [doc['filename']for doc in vector_results[:5]]
print("=== TOP 5 TEXT SEARCH FILENAMES ===")
for i, filename in enumerate(top_text_files, 1):
    print(f"{i}. {filename}")

print("\n=== TOP 5 VECTOR SEARCH FILENAMES ===")
for i, filename in enumerate(top_vector_files, 1):
    print(f"{i}. {filename}")


unique_to_vector = set(top_vector_files) - set(top_text_files)
print("\n👉 File in vector results but NOT text results:", unique_to_vector)

=== TOP 5 TEXT SEARCH FILENAMES ===
1. 02-vector-search/lessons/02-embeddings.md
2. 03-orchestration/lessons/05-rag.md
3. 02-vector-search/lessons/01-intro.md
4. 03-orchestration/lessons/05-rag.md
5. 02-vector-search/lessons/01-intro.md

=== TOP 5 VECTOR SEARCH FILENAMES ===
1. 02-vector-search/lessons/08-pgvector.md
2. 02-vector-search/lessons/08-pgvector.md
3. 03-orchestration/lessons/05-rag.md
4. 02-vector-search/lessons/08-pgvector.md
5. 02-vector-search/lessons/08-pgvector.md

👉 File in vector results but NOT text results: {'02-vector-search/lessons/08-pgvector.md'}


Q6. Hybrid search

In [20]:
from minsearch import VectorSearch,Index

# --- 1. TEXT SEARCH SETUP ---
text_index = Index(text_fields = ['content'],keyword_fields = ['course'])
text_index.fit(chunks)
text_results = text_index.search(query ="How do I give the model access to tools?",
                             filter_dict = {"course": "llm-zoomcamp"},
                             num_results = 5)

# --- 2. VECTOR SEARCH SETUP ---
vindex = VectorSearch(keyword_fields = ['course'])
vindex.fit(X,chunks)

query_vector = model.encode("How do I give the model access to tools?")

vector_results = vindex.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)


In [22]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [23]:
results = rrf([vector_results, text_results])

In [18]:
type(results)

list

In [24]:
results[0]

{'start': 4000,
 'content': ' function. `parameters` is a JSON schema\nfor the arguments, and we mark `query` as required so the model always\nfills it in.\n\n## Sending the question with the tool\n\nNow we send the same question as before, but this time we include the\ntool in the request:\n\n```python\nresponse = openai_client.responses.create(\n    model="gpt-5.4-mini",\n    input=messages,\n    tools=[search_tool],\n)\n\nresponse.output\n```\n\nLook at the output. Instead of a message with the answer, the response\ncontains a `function_call` entry. The model decided it needs to search\nthe FAQ before answering. Rather than reply, it asked us to run the\nsearch function first.\n\nLook at the arguments too. The model didn\'t pass our question\nverbatim. It judged the raw question wasn\'t the best query to search\nwith. So it rewrote our enrollment question into search keywords like\n"enroll late join course".\n\n## Executing the function and sending the result back\n\nThe function ca